In [1]:
import qutip as qt
from qutip import qeye, tensor, basis, Qobj
import numpy as np
from scipy.linalg import fractional_matrix_power

This will not work if you have mixed state but for pure states without any error channels this will suffice

In [2]:
cnot_operations_set = [[basis(2, 0) * basis(2, 0).dag(), qeye(2), qeye(2)], [basis(2, 1) * basis(2, 1).dag(), Qobj([[0, 1], [1, 0]]), qeye(2)]]
qubit_count = 3
a = 1
b = 0
rho = [(a * basis(2, 0) + b * basis(2, 1)).unit() * (a * basis(2, 0) + b * basis(2, 1)).unit().dag(), basis(2, 1) * basis(2, 1).dag(), basis(2, 1) * basis(2, 1).dag()]

In [3]:
# without trotterization:
rho_new = [[qeye(2) for _ in range(qubit_count)] for _ in range(len(cnot_operations_set))]
for i in range(len(cnot_operations_set)):
    for j in range(qubit_count):
        rho_new[i][j] = cnot_operations_set[i][j] * rho[j] * cnot_operations_set[i][j]

# before summing you would have to tensor the results first 
# successful

In [4]:
# creating the cnot
unitary = []
for i in range(len(cnot_operations_set)):
    new_unitrary = [qeye(2) for _ in range(qubit_count)]
    for j in range(qubit_count):
        new_unitrary[j] = Qobj(fractional_matrix_power(cnot_operations_set[i][j], 1/2))
    unitary.append(new_unitrary)
# successful

In [5]:
# apply the cnot in a trotterized manner 
new_rho = []

for i in range(len(unitary)):
    rho_exc = [rho[j] for j in range(qubit_count)]
    for j in range(qubit_count):
        for k in range(2): # the two here is the number of trotter steps 
            rho_exc[j] = unitary[i][j] * rho_exc[j] * unitary[i][j].dag()
    new_rho.append(rho_exc)

    
sum([tensor(new_rho[i]) for i in range(len(new_rho))])
# successful

Quantum object: dims = [[2, 2, 2], [2, 2, 2]], shape = (8, 8), type = oper, isherm = True
Qobj data =
[[0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]]

In [6]:
# create two gates 
# pass in an array of arrays
x_gate = [[Qobj([[0, 1], [1, 0]]), qeye(2), qeye(2)]]
unitary1 = []
for i in range(len(x_gate)):
    new_unitrary = [qeye(2) for _ in range(qubit_count)]
    for j in range(qubit_count):
        new_unitrary[j] = Qobj(fractional_matrix_power(x_gate[i][j], 1/2))
    unitary1.append(new_unitrary)
gate_set = [unitary1, unitary]
# successful

In [7]:
# import copy
# from qutip import Qobj, basis, qeye

# # Defining the quantum gates
# x_gate = [[Qobj([[0, 1], [1, 0]]), qeye(2)]]
# cnot_operations_set = [
#     [basis(2, 0) * basis(2, 0).dag(), qeye(2)],  # CNOT for |0> state
#     [basis(2, 1) * basis(2, 1).dag(), Qobj([[0, 1], [1, 0]])]  # CNOT for |1> state
# ]

# # Aligning matrices in gate_set to the same shape (e.g., 2x2 matrices)
# for i in range(len(gate_set) - 1):
#     if len(gate_set[i]) < len(gate_set[i + 1]):
#         # Resize gate_set[i] to match gate_set[i + 1]
#         gate_set[i] = [gate_set[i][0]] * (len(gate_set[i + 1]) // len(gate_set[i]))
#     elif len(gate_set[i]) > len(gate_set[i + 1]):
#         # Resize gate_set[i + 1] to match gate_set[i]
#         gate_set[i + 1] = [gate_set[i + 1][0]] * (len(gate_set[i]) // len(gate_set[i + 1]))


# # Deep copy the first matrix to initialize result
# result = copy.deepcopy(gate_set[0])


# # Element-wise multiplication of matrices in gate_set
# for i in range(1, len(gate_set)):

#     # Check that gate_set[i] and result are of the same size
#     if len(result) != len(gate_set[i]):
#         print(f"Error: Dimensions of result and gate_set[{i}] do not match!")
#         continue

#     for j in range(len(gate_set[i])):
#         for k in range(len(gate_set[i][j])):
#             # Debugging: Print current elements being multiplied
#             print(f"Multiplying: result[{j}][{k}] = {result[j][k]} * gate_set[{i}][{j}][{k}] = {gate_set[i][j][k]}")
            
#             # Perform element-wise multiplication (Qobj multiplication)
#             result[j][k] = gate_set[i][j][k] * result[j][k]

#             # Debugging: Print updated result
#             print(f"Updated result[{j}][{k}] = {result[j][k]}")

# # # Final result check
# # print(f"\nFinal result after all multiplications:")
# # for row in result:
# #     print([str(g) for g in row])


In [8]:
import copy
hada_layer = [[(1/np.sqrt(2)) * Qobj([[1, 1], [1, -1]]), qeye(2), qeye(2)]]
gate_set = [hada_layer, cnot_operations_set]
# gate_set = [cnot_operations_set]
# gate_set = [hada_layer]

# gate_set
# can you get around this by just combining a single qubit gate into two?
# sorting to make sure that they are the same size
for i in range(len(gate_set) - 1):
    if len(gate_set[i]) < len(gate_set[i+1]):
        gate_set[i] = ([gate_set[i][0]] * int((len(gate_set[i+1]) / len(gate_set[i]))))
    elif len(gate_set[i]) > len(gate_set[i+1]):
        gate_set[i+1] = ([gate_set[i+1][0]] * int((len(gate_set[i]) / len(gate_set[i+1]))))

# now you need to do element wise multiplication and then you have the gate that is applied to each individual qubit and then summed at the end 
# Initialize result with copies of the Qobj matrices from gate_set[0]
result = [[copy.deepcopy(g) for g in row] for row in gate_set[0]]


# this makes it into one effective gate 
for i in range(1, len(gate_set)):
    for j in range(len(gate_set[i])):
        for k in range(len(gate_set[i][j])):
            result[j][k] = gate_set[i][j][k] * result[j][k]


# apply the gate to the state 
rho = [basis(2, 1) * basis(2, 1).dag(), basis(2, 0) * basis(2, 0).dag()]
rho = [basis(2, 1) , basis(2, 0), basis(2, 1)]

# without trotterization:
# rho_new = [[qeye(2) for _ in range(qubit_count)] for _ in range(len(result))]
rho_new = [[copy.deepcopy(qubit) for qubit in rho] for _ in range(len(result))]
# rho_new_full = []
for i in range(len(result)):
    # rho_new = []
    for j in range(qubit_count):
        # rho_new[i][j] = result[i][j] * rho_new[i][j] * result[i][j].dag()
        rho_new[i][j] = result[i][j] * rho_new[i][j]
        # rho_new.append(result[i][j] * rho[j] * result[i][j])
    # rho_new_full.append(rho_new)

sum([tensor(rho_res) for rho_res in rho_new]) / sum([tensor(rho_res) for rho_res in rho_new]).norm() * (sum([tensor(rho_res) for rho_res in rho_new]) / sum([tensor(rho_res) for rho_res in rho_new]).norm()).dag()

# this fully work but it seems weird for entanglement 

Quantum object: dims = [[2, 2, 2], [2, 2, 2]], shape = (8, 8), type = oper, isherm = True
Qobj data =
[[ 0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 0.   0.5  0.   0.   0.   0.   0.  -0.5]
 [ 0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 0.  -0.5  0.   0.   0.   0.   0.   0.5]]

In [9]:
rho = [(a * basis(2, 0) + b * basis(2, 1)).unit() * (a * basis(2, 0) + b * basis(2, 1)).unit().dag(), basis(2, 1) * basis(2, 1).dag()]
tensor(rho)

Quantum object: dims = [[2, 2], [2, 2]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[0. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

In [10]:
gate_set[0][0][0]

Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
Qobj data =
[[ 0.70710678  0.70710678]
 [ 0.70710678 -0.70710678]]

In [11]:
gate_set[1]

[[Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
  Qobj data =
  [[1. 0.]
   [0. 0.]],
  Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
  Qobj data =
  [[1. 0.]
   [0. 1.]],
  Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
  Qobj data =
  [[1. 0.]
   [0. 1.]]],
 [Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
  Qobj data =
  [[0. 0.]
   [0. 1.]],
  Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
  Qobj data =
  [[0. 1.]
   [1. 0.]],
  Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
  Qobj data =
  [[1. 0.]
   [0. 1.]]]]

In [12]:
gate_set[0] = [gate_set[0]] * 2
# for i in range(len(gate_set) - 1):
#     i = i + 1
#     if len(gate_set[i]) == len(result):
#         result = np.array(gate_set[i]) * result
#     elif len(gate_set[i]) < len(result):
#         mat_new = np.tile(gate_set[i], (2,1))
#         result = np.array(gate_set[i]) * result
#     elif len(gate_set[i]) > len(result):
#         mat_new = np.tile(result, (2,1))
#         result = np.array(gate_set[i]) * result

gate_set[0] 

[[[Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
   Qobj data =
   [[ 0.70710678  0.70710678]
    [ 0.70710678 -0.70710678]],
   Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
   Qobj data =
   [[1. 0.]
    [0. 1.]],
   Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
   Qobj data =
   [[1. 0.]
    [0. 1.]]],
  [Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
   Qobj data =
   [[ 0.70710678  0.70710678]
    [ 0.70710678 -0.70710678]],
   Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
   Qobj data =
   [[1. 0.]
    [0. 1.]],
   Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
   Qobj data =
   [[1. 0.]
    [0. 1.]]]],
 [[Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
   Qobj data =
   [[ 0.70710678  0.70710678]
    [ 0.70710678 -0.70710678]],
   Quantum object: dims = [[2], [2]

In [13]:
matrix = [[[1, 2]], [[4, 7], [9, 11]], [2, 3]]
# retiling the matrices to make them have the same shape but it will be the same result in the end 
matrix_a_tiled = np.tile(matrix[0], (2,1))
results = matrix_a_tiled * matrix[1]
result = np.array(matrix[0])
for i in range( len(matrix) - 1):
    i = i + 1
    if len(matrix[i]) == len(result):
        result = matrix[i] * result
    elif len(matrix[i]) < len(result):
        mat_new = np.tile(matrix[i], (2,1))
        result = matrix[i] * result
    elif len(matrix[i]) > len(result):
        mat_new = np.tile(result, (2,1))
        result = np.array(matrix[i]) * np.array(result)

result

array([[ 8, 42],
       [18, 66]])

In [14]:
results = []
for array in range(len(matrix)):
    if array != len(matrix) - 1:
        for index in len(matrix[array]):
            if index != len(matrix[array]) - 1:
                results.append(matrix[array][index] * matrix[array + 1][index])
            else:
                


IndentationError: expected an indented block (2910993175.py, line 8)

In [ ]:
import math
qubit_ops = {}

for i in range(len(matrix)):
    for j in range(len(matrix[i])):
        for k in range(qubit_count):
            qubit_ops.update({f"{i}{j}{k}": matrix[i][j][k]})

qub_op = [[1 for _ in range(qubit_count)]] * math.prod([len(mat) for mat in matrix])
qub_dict = {}
for i in range(math.prod([len(mat) for mat in matrix])):
    int_dict = {}
    for j in range(qubit_count):
        int_dict.update({f"{i}{j}": 1})
    qub_dict.update(int_dict)

# now since a dictionary is made with all of the values multiply the results 
for point in qubit_ops:
    if point[0] == qubit_ops[0] and 


qubit_ops

In [ ]:
for string in qubit_operator:
    print(string)

# if len(matrix) > 1:        
#     for i in qubit_ops:
#         first_num = str(i)[0]
#         second_num = str(i)[1]
#         third_num = str(i)[2]

#         j_val = qubit_operator[]

# qubit_operator

what happens if you multiply the individual parts of the parallel gates together first before applying it to the state?

In [ ]:
gates = []
for k in range(qubit_count):
    new_gate = []
    gate = qeye(2)
    for i in range(len(gate_set)): 
        for j in range(len(gate_set[i])):
            gate = gate_set[i][j][k] * gate
            
    new_gate.append(gate)
gates.append(new_gate)
        
gates

In [ ]:
# trotterization parallel execution
new_rho = []
for i in range(len(gate_set)):
    for j in range(len(gate_set[i])):
        rho_exc = [rho[j] for j in range(qubit_count)]
        for k in range(qubit_count):
            rho_exc[j] = gate_set[i][j][k] * rho_exc[j] * gate_set[i][j][k].dag()
        new_rho.append(rho_exc)

sum([tensor(new_rho[i]) for i in range(len(new_rho))])

In [ ]:
# applying gates in parallel if they are cnots or not 
new_rho = []
# for h in range(len(gate_set)):
for k in range(2): # the two here is the number of trotter steps 
    rho_exc = [rho[j] for j in range(qubit_count)]
    for h in range(len(gate_set)):

        for i in range(len(gate_set[i])):

            for j in range(qubit_count):
                rho_exc[j] = gate_set[h][i][j] * rho_exc[j] * gate_set[h][i][j].dag()
    new_rho.append(rho_exc)

    
sum([tensor(new_rho[i]) for i in range(len(new_rho))])